# Quickstart：加载预训练权重 → 直接回测

10 分钟内看到结果，不需要训练。

**前置条件**
- 已准备好 `CSI100_stock_data_with_factors.pkl`（放在项目根目录）
- 已从 Releases 下载 `pretrained/itransformer_morl.pth`
- 已完成环境安装：`conda env create -f environment.yml`

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # 指向项目根目录

import torch
import yaml

from graphrl.data.loader import load_real_data
from graphrl.data.dataset import create_enhanced_dataset
from graphrl.models.edygformer import E_DyGFormer
from graphrl.rl.policy import PolicyNet, PolicyModelAdapter
from graphrl.backtest.engine import EnhancedBacktestEngine
from graphrl.utils.plotting import plot_enhanced_results
from graphrl.utils.seed import set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

In [ ]:
# 加载配置
with open('../configs/experiments/itransformer_morl.yaml') as f:
    cfg = yaml.safe_load(f)

# 数据路径（修改为你的实际路径）
cfg['data_file_path'] = '../CSI100_stock_data_with_factors.pkl'

print('配置加载完成')

In [ ]:
# 加载数据
price, bench, panel = load_real_data(
    cfg['data_file_path'],
    max_assets=cfg['max_assets'],
    max_days=cfg['max_days'],
    index_data_path='../CSI100_index.pkl',   # CSI100 指数基准
)
dataset, feat_names = create_enhanced_dataset(
    panel,
    seq_len=cfg['seq_len'],
    corr_window=cfg['corr_window'],
    corr_threshold=cfg['corr_threshold'],
)

n_all      = len(dataset)
train_size = int(cfg['train_ratio'] * n_all)
val_size   = int(cfg['val_ratio']   * n_all)
val_ds     = dataset[train_size:train_size + val_size]
test_ds    = dataset[train_size + val_size:]
print(f'测试集大小: {len(test_ds)} 天')

In [ ]:
# 构建模型并加载预训练权重
feat_dim   = dataset[0].x.shape[1]
num_assets = len(price.columns)

enc_cfg = dict(cfg['encoder'])
enc_cfg.update({'node_feat_dim': feat_dim, 'num_assets': num_assets, 'seq_len': cfg['seq_len']})
encoder = E_DyGFormer(**enc_cfg)
policy  = PolicyNet(encoder, **cfg['policy']).to(device)

checkpoint_path = '../pretrained/itransformer_morl.pth'
policy.load_state_dict(torch.load(checkpoint_path, map_location=device))
policy.eval()
print(f'权重加载成功: {checkpoint_path}')

In [ ]:
# 回测（测试集）
adapter = PolicyModelAdapter(policy, smooth_weights=True, smooth_rho=0.3)
engine  = EnhancedBacktestEngine(**cfg['engine'])

price_test = price.iloc[-len(test_ds):]
bench_test = bench.iloc[-len(test_ds):]

res = engine.run_backtest(adapter, test_ds, price_test, bench_test,
                          temperature=policy.temperature)

print(f"\n=== 测试集结果 ===")
print(f"年化收益: {res['annual_return']:.2%}")
print(f"夏普比率: {res['sharpe_ratio']:.3f}")
print(f"信息比率: {res['information_ratio']:.3f}")
print(f"最大回撤: {res['max_drawdown']:.2%}")
print(f"年化超额: {res['excess_return']:.2%}")

In [ ]:
# 净值曲线可视化
plot_enhanced_results(res, engine.portfolio_value, engine.benchmark_levels)